# 21 Sep

## Idea here is to make 4 scripts that will be capable of taking bulk csv in a normalized format that will then be able to run and automatically make data into a wanddb friendly format and standardise it


## Individual compound normaliser
Expected inputs:
- bulk inchi list from wand (required): filepath
- name : string
- compound_source : string
- smiles/inchi/iupac (at least 1 required) : string
- 

In [ ]:
import pandas as pd
from rdkit import Chem
import nispo
from Pyopsin import OpSin

wand_inchis_path = ''
name = ''
source = ''
smiles = ''
inchi = ''
iupac = ''



## bulk compound normaliser
not going to make it too interoperable just yet but for now I will make it so that it can use a file with inchis

- Expected inputs: 
- bulk inchi list from wand with names, synonyms, iupac and pubchemids (required): filepath
- compound_csv (column names as stated): 
- 'name' : string
- 'compound_source' : string
- 'smiles'/'inchi'/'iupac' (at least 1 required) : string

In [ ]:
# wand compounds expected headings ['wand_id','inchi','name','synonyms','iupac']
# new compounds expected headings ['inchi','name','synonyms','iupac']
import pandas as pd
from rdkit import Chem
import time
import pubchempy as pcp
from nispo import mol_to_iupac

#//////////////////////////////////////////////////////paths
wand_inchis_path = ''
new_compounds_path = ''
error_output_path = ''
output_path = ''
report_path = ''
#//////////////////////////////////////////////////////data
wand_compounds = pd.read_csv(wand_inchis_path)
new_compounds = pd.read_csv(new_compounds_path)
origional_size = new_compounds.shape[0]
errors = pd.DataFrame(columns = ['id','type','error_msg'])
#//////////////////////////////////////////////////////functions
def get_smiles2D(inchi):
    global errors

    try:
        mol = Chem.MolFromInchi(inchi, treatWarningAsError=True)
        if mol:
            c_smiles = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=False)
            return c_smiles
        else:
            err = [inchi,'inchi','Could not convert inchi to 2D smiles']
            errors = errors.concat(err)
            return None
    except Exception as e:
        err = [inchi,'inchi',str(e)]
        errors.append(err)
        return None, 

def get_smiles3D(inchi):
    global errors
    try:
        mol = Chem.MolFromInchi(inchi, treatWarningAsError=True)
        if mol:
            i_smiles = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
            return i_smiles
        else:
            err = [inchi,'inchi','Could not convert inchi to isomericsmiles']
            errors = errors.concat(err)
            return None
    except Exception as e:
        err = [inchi,'inchi',str(e)]
        errors.append(err)
        return None

def get_cid(input,type):
    global errors
    for k in range(5):
        try:
            compound = pcp.get_compounds(input,type)
            if compound:
                cid = compound[0].cid
                return cid
            else:
                err = [input,type,'Does not appear to be in pubchem with this inchi']
                errors.append(err)
                return None
        except Exception as e:
            error_msg = str(e)
            if '502' in error_msg or 'bad gateway' in error_msg.lower() or 'timeout' in error_msg.lower():
                time.sleep(0.2)
            else:
                err = [input,type,str(e)]
                errors.append(err)
                return None
    err = [input,type,'retry not sucessful']
    errors.append(err)
    return None

def get_inchikey(inchi):
    global errors
    try:
        mol = Chem.MolFromInchi(inchi, treatWarningAsError=True)
        if mol:
            key = Chem.MolToInchiKey(mol)
            return key
        else:
            err = [inchi,'inchi','Could not convert inchi to inchi key']
            errors.append(err)
            return None
    except Exception as e:
        err = [inchi,'inchi',str(e)]
        errors.append(err)
        return None

def get_iupac(inchi):
    global errors
    try:
        mol = Chem.MolFromInchi(inchi,treatWarningAsError=True)
        if mol:
            iupac = mol_to_iupac(mol)
            if pd.notna(iupac):
                return iupac
        else:
          err = [inchi,'inchi','Could not convert inchi to inchi key']
          errors = errors.concat(err)
          return None  
    except Exception as e:
        err = [inchi,'inchi',str(e)]
        errors.append(err)
        return None         
        

#//////////////////////////////////////////////////////main
                        #//////////////////////////////check if inchis are missing or if are already in the DB and remove from new add to err and add as new metadata
wand_set = set(wand_compounds['inchi'].tolist())

if new_compounds['inchi'].isna().any():
    missing_inchi = pd.DataFrame(columns=['id','type','error_msg'])
    missing_inchi['id'] = new_compounds['smiles'][new_compounds['inchi'].isna()]
    missing_inchi['type'] = 'smiles'
    missing_inchi['error_msg'] = 'missing an inchi'
    errors = pd.concat([errors,missing_inchi], ignore_index=True)
new_compounds.dropna(subset=['inchi'], inplace=True)

new_inchis = set(new_compounds['inchi'].tolist())
in_db = pd.DataFrame(columns=['id','type','error_msg'])
in_db['id'] = new_compounds[new_compounds['inchi'].isin(wand_set)]['inchi'].to_list()
in_db['type'] = 'inchi'
in_db['error_msg'] = 'inchi already in DB'
errors = pd.concat([errors,in_db], ignore_index=True)
compounds_already_in_db = new_compounds[new_compounds['inchi'].isin(wand_set)]
new_compounds = new_compounds[~new_compounds['inchi'].isin(wand_set)]

                        #//////////////////////////////add new metadata to compounds already in DB
#if compounds_already_in_db:
 #   pass
                        #//////////////////////////////add standardized smiles
new_compounds['canonical_smiles'] = new_compounds['inchi'].apply(get_smiles2D)
new_compounds['isomeric_smiles'] = new_compounds['inchi'].apply(get_smiles3D)
                        #//////////////////////////////add cid
new_compounds['cid'] = new_compounds['inchi'].apply(get_cid,'inchi')
                        #//////////////////////////////Add iupac
new_compounds['iupac'] = new_compounds['inchi'].apply(get_iupac).where(new_compounds['iupac'].isna())
                        #//////////////////////////////Add inchikey
new_compounds['iupac'] = new_compounds['inchi'].apply(get_iupac).where(new_compounds['iupac'].isna())

#//////////////////////////////////////////////////////report
new_size = new_compounds.shape[0]
errors_size = errors.shape[0]

items_removed = origional_size - new_size

error_types = errors['error_msg'].value_counts(dropna=False)

item = f"Compounds in starting file: {origional_size}\n\
         New compounds standardised to add to the database: {new_size}\n\
         Removed compounds due to errors: {items_removed}\n\
         Errors: {errors_size}\n\
         Error types:\n\
         {error_types}\n\
         "

with open(report_path, 'w') as f:
    f.write(item)
#//////////////////////////////////////////////////////output
new_compounds.to_csv(output_path, index=False)
errors.to_csv(error_output_path, index=False)